In [3]:

# HOSPITAL READMISSION PREDICTION
# Logistic Regression with L2 Regularization


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score
)


# 1. Load dataset


file_path = "/content/diabetic_data.csv"

data = pd.read_csv(file_path)

print("Original dataset shape:", data.shape)


# 2. Separate features and target


X = data.drop(columns=["readmitted"])
y = data["readmitted"]

print("\nOriginal target distribution:")
print(y.value_counts())


# 3. Create binary target


# <30  = readmitted within 30 days = 1
# Other = not readmitted within 30 days = 0

y = (y == "<30").astype(int)

print("\nBinary target distribution:")
print(y.value_counts())

print("\nBinary target percentage:")
print((y.value_counts(normalize=True) * 100).round(2))


# 4. Replace missing value markers


X = X.replace("?", np.nan)


# 5. Remove ID columns


X = X.drop(
    columns=["encounter_id", "patient_nbr"],
    errors="ignore"
)


# 6. Identify feature types


numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


# 7. Preprocessing


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


# 8. Train-test split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# 9. Logistic Regression + L2 Regularization


model = Pipeline([
    ("preprocessor", preprocessor),

    ("classifier", LogisticRegression(
        penalty="l2",
        C=1.0,
        max_iter=2000,
        solver="liblinear"
    ))
])


# 10. Train model


model.fit(X_train, y_train)

print("\nModel training completed.")

Original dataset shape: (101766, 50)

Original target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Binary target distribution:
readmitted
0    90409
1    11357
Name: count, dtype: int64

Binary target percentage:
readmitted
0    88.84
1    11.16
Name: proportion, dtype: float64

Numerical features: 11
Categorical features: 36

Training samples: 81412
Testing samples: 20354

Model training completed.


In [4]:
# ============================================
# 11. Predictions
# ============================================

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

# ============================================
# 12. Evaluation
# ============================================

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("\n======================================")
print("MODEL PERFORMANCE")
print("======================================")

print(f"Accuracy     : {accuracy:.4f}")
print(f"ROC-AUC      : {auc:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"Sensitivity  : {sensitivity:.4f}")
print(f"Specificity  : {specificity:.4f}")

print("\n======================================")
print("CONFUSION MATRIX")
print("======================================")

print(confusion_matrix(y_test, y_pred))

print("\n======================================")
print("CLASSIFICATION REPORT")
print("======================================")

print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "No readmission",
        "Readmission within 30 days"
    ]
))


MODEL PERFORMANCE
Accuracy     : 0.8883
ROC-AUC      : 0.6464
Precision    : 0.4884
Recall       : 0.0185
Sensitivity  : 0.0185
Specificity  : 0.9976

CONFUSION MATRIX
[[18039    44]
 [ 2229    42]]

CLASSIFICATION REPORT
                            precision    recall  f1-score   support

            No readmission       0.89      1.00      0.94     18083
Readmission within 30 days       0.49      0.02      0.04      2271

                  accuracy                           0.89     20354
                 macro avg       0.69      0.51      0.49     20354
              weighted avg       0.85      0.89      0.84     20354

